# Capability 19: Multiple KPIs, entities, dimensions, and hierarchical business structures

10/10 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 19`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap19_multi_kpi_entity_hierarchy.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: revenue KPI

**✅ PASS**

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

## 02: volume KPI

**✅ PASS**

In [ ]:
_ = ask("What was EMEA's volume in Q2 2025?", '02')

02
USER: What was EMEA's volume in Q2 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s volume in Q2 2025 was **24,172.0 thousand hectoliters (hL)**.

If you’d like, I can help pull the year-over-year change or a country/brand breakdown for the same period.

## 03: normalized EBITDA KPI

**✅ PASS**

In [ ]:
_ = ask("What was Asia Pacific's normalized EBITDA in 2025?", '03')

03
USER: What was Asia Pacific's normalized EBITDA in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: Asia Pacific's normalized EBITDA for 2025 was **1,700.0 USD million** (i.e., **$1.7 billion**), based on the sum of quarterly figures.

If you'd like, I can break this down by quarter or compare it against 2024/2023 to show the trend.

## 04: EBITDA margin KPI (computed)

**✅ PASS**

In [ ]:
_ = ask("What was South America's EBITDA margin in Q4 2025?", '04')

04
USER: What was South America's EBITDA margin in Q4 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: South America’s EBITDA margin in Q4 2025 was **36.2%**.

## 05: organic revenue growth KPI

**✅ PASS**

In [ ]:
_ = ask("What was Middle Americas' organic revenue growth in 2025?", '05')

05
USER: What was Middle Americas' organic revenue growth in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: **Middle Americas' organic revenue growth for 2025 could not be retrieved.**

The structured KPI query for zone = Middle Americas, year = 2025 (annual grain) returned no value — the result cell was empty, indicating the figure is missing or not yet available in the data. I can't report a number here, and I won't estimate or invent one.

Possible reasons include the metric not yet being loaded for 2025, a different grain (e.g., quarterly) being available, or the zone/metric being recorded under a different name.

If helpful, I could next:

- Check Middle Americas' organic revenue growth for a prior year (e.g., 2024) to establish a baseline.
- Look for 2025 figures for other zones as a comparison.
- Verify whether the metric exists under quarterly grain or a different KPI name.

## 06: net profit KPI

**✅ PASS**

In [ ]:
_ = ask("What was AB InBev's net profit in FY2025?", '06')

06
USER: What was AB InBev's net profit in FY2025?

[intent=data_query | sub_agents=['structured']]

AGENT: AB InBev’s **net profit for FY2025 (Global)** was **$6,837 million USD**.

This figure comes directly from the structured KPI data for Global, full-year 2025.

If useful, I can compare this against FY2024 net profit or break down profitability by region/segment.

## 07: hierarchy: country resolves under its zone

**✅ PASS**

In [ ]:
_ = ask('What was the revenue for the United States in Q1 2024?', '07')

07
USER: What was the revenue for the United States in Q1 2024?

[intent=data_query | sub_agents=['structured']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **North America** zone (which includes United States) instead, plus any United States-specific commentary found in the documents.

AGENT: AB InBev does not publicly disclose standalone U.S. revenue. Based on the available structured data, the closest figure is for the **North America** zone (which includes the United States):

| Zone | Year | Quarter | Revenue (USD million) |
| --- | --- | --- | --- |
| North America | 2024 | 1 | $3,593 |

So the **North America zone revenue for Q1 2024 was $3,593 million (~$3.59 billion)**. A United States–only figure was not available in the retrieved evidence.

If helpful, I can compare this to Q1 2023 or look for U.S.-specific commentary in the internal documents.

## 08: hierarchy: a different country under a different zone

**✅ PASS**

In [ ]:
_ = ask("What was Colombia's revenue contribution in 2025?", '08')

08
USER: What was Colombia's revenue contribution in 2025?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-011', 'DOC-012', 'DOC-004', 'DOC-008', 'DOC-006']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **Middle Americas** zone (which includes Colombia) instead, plus any Colombia-specific commentary found in the documents.

AGENT: ## Colombia revenue contribution, FY2025

**Short answer: AB InBev does not disclose revenue by country, so a Colombia-only revenue figure is not available.** The query on `country = 'Colombia'` returns the zone-level roll-up instead:

| Zone | Year | Revenue (USD million) |
| --- | --- | --- |
| Middle Americas | 2025 | $17,376 |

So the defensible statement is: **the Middle Americas zone — which includes Colombia — generated $17,376M of revenue in 2025.** That is a zone total, not Colombia's contribution, and no country-level split is published tha

## 09: entity dimension: brand-level qualitative question

**✅ PASS**

In [ ]:
_ = ask('How is Stella Artois performing globally?', '09')

09
USER: How is Stella Artois performing globally?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-013', 'DOC-015', 'DOC-010']]

AGENT: Based on the evidence retrieved, I don’t have a brand-specific performance figure for Stella Artois itself. The available data doesn’t include a Stella Artois–level revenue, volume, or EBITDA line.

What the evidence does show:

- Stella Artois is grouped with AB InBev’s “megabrands,” alongside Budweiser, Corona, and Michelob Ultra [DOC-012].
- These megabrands collectively grew revenue **4.6% in FY2024** and **4.1% in FY2025**, with both years ahead of total company revenue growth [DOC-012].
- However, no document in the retrieved set breaks out Stella Artois individually, and the structured database covers zone/company-level results rather than brand-level performance [DOC-015].

So the honest answer is: **globally, Stella Artois appears to be part of a megabrand portfolio that is growing slightly ahead of t

## 10: multiple dimensions combined: zone + KPI + period in one ask

**✅ PASS**

In [ ]:
_ = ask("What was North America's EBITDA margin in Q1 2024?", '10')

10
USER: What was North America's EBITDA margin in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's EBITDA margin in Q1 2024 was **31.3%**.